In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv('vgsales.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 1.4 MB


In [3]:
df['Genre'].unique()

<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy']
Length: 12, dtype: str

In [4]:
df['Platform'].unique()

<StringArray>
[ 'Wii',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  '3DS',  'PS4',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XOne',
   'GC', 'WiiU',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str

In [5]:
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [6]:
df['Year'] = df['Year'].astype('Int64')
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [7]:
df.describe()

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.0,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.0,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.0,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.0,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.0,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.0,41.490000,29.020000,10.220000,10.570000,82.740000


In [8]:
df.isnull().sum()

Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

In [9]:
df["Publisher"] = df["Publisher"].fillna("Unknown")
df = df.dropna(subset=["Genre", "Platform", "Name"]).reset_index(drop=True)

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  Int64  
 4   Genre         16598 non-null  str    
 5   Publisher     16598 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: Int64(1), float64(5), int64(1), str(4)
memory usage: 1.4 MB


In [11]:
#one hot encoding
sales_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
genre_dummies = pd.get_dummies(df["Genre"], prefix="genre")
platform_dummies = pd.get_dummies(df["Platform"], prefix="platform")
publisher_dummies = pd.get_dummies(df["Publisher"], prefix="pub")

In [12]:
#normalizing the sales figure so that large values wont dominate similarity
scaler = MinMaxScaler()
sales_scaled = pd.DataFrame(
        scaler.fit_transform(df[sales_cols]),
        columns=sales_cols,
        index=df.index,
    )
#merge all sales into one single matrix for similarity calculations
features = pd.concat([genre_dummies, platform_dummies, publisher_dummies, sales_scaled], axis=1)

In [13]:
#filter games based on users criteria
def filter_games(df, genre: str, platform: str, max_rank: int, top_n: int):
    result = df.copy()
    #lowering all the elements on criteria given by user and also the dataset 
    if genre:
        result = result[result["Genre"].str.lower() == genre.lower()]
    if platform:
        result = result[result["Platform"].str.lower() == platform.lower()]
    if max_rank:
        result = result[result["Rank"] <= max_rank]
    #grab top n number of games sorted by ranks
    result = result.sort_values("Rank").head(top_n)
    return result

In [14]:
#
def find_similar_games(df, features, seed_indices: list, top_n: int):
    if len(seed_indices) == 0:
        return pd.DataFrame()

    seed_vectors = features.loc[seed_indices]
    sim_matrix = cosine_similarity(seed_vectors, features)

    avg_sim = sim_matrix.mean(axis=0)

    sim_series = pd.Series(avg_sim, index=features.index, name="similarity")
    sim_series = sim_series.drop(index=seed_indices, errors="ignore")

    top_matches = sim_series.sort_values(ascending=False).head(top_n)

    result = df.loc[top_matches.index].copy()
    result["similarity"] = top_matches.values
    return result.sort_values("similarity", ascending=False)

In [15]:
def recommend(df, features, genre: str, platform: str, max_rank: int, top_n_filtered: int = 5, top_n_similar: int = 5):
    filtered = filter_games(df, genre, platform, max_rank, top_n=top_n_filtered)

    print(f"\n===Top {len(filtered)} games recommended for you as per your genre and platform===")
    if filtered.empty:
        print("No games matched those filters.")
    else:
        print(filtered[["Rank", "Name", "Platform", "Genre", "Global_Sales"]].to_string(index=False))

    similar = find_similar_games(df, features, seed_indices=filtered.index.tolist(), top_n=top_n_similar)

    print(f"\n=== Games similar to the above (content based) ===")
    if similar.empty:
        print("No similar games found.")
    else:
        print(similar[["Name", "Platform", "Genre", "Global_Sales", "similarity"]].to_string(index=False))

    return filtered, similar

In [16]:
if __name__ == "__main__":
    df

    recommend(
        df,
        features,
        genre="adventure",
        platform="nes",
        max_rank=500,
        top_n_filtered=5,
        top_n_similar=5,
    )


===Top 1 games recommended for you as per your genre and platform===
 Rank                            Name Platform     Genre  Global_Sales
  252 Zelda II: The Adventure of Link      NES Adventure          4.38

=== Games similar to the above (content based) ===
               Name Platform  Genre  Global_Sales  similarity
             Tetris      NES Puzzle          5.58    0.670480
               Golf      NES Sports          4.01    0.670469
The Legend of Zelda      NES Action          6.51    0.670316
           Baseball      NES Sports          3.20    0.670217
          Dr. Mario      NES Puzzle          4.85    0.669972
